# run_e44
Phase 3 CPU Track Stage A2 Worker 3 (`E44`): selective-confidence coverage & retained risk at nominal `{50%, 75%, 90%}` across `last_caption:raw`, `last_caption:source_dev_c`, `last_caption:target_dev_c`, `caption_mean:raw`, and `caption_mean:e11_mean_scale`.

- Run on Colab CPU (`requirements-stage2-cpu.txt`, Python 3.11/3.12, single BLAS/OpenMP thread).
- Edit only Cell 3 (`MODEL = 'qwen'` or `'gemma'`, `PLAN_REF`, optional `RESUME_ANALYSIS_REF`).
- Writes only under `PHASE1_ROOT/phase3/results/e44/<model>/`; Phase 1 & Phase 2 remain strictly read-only; `CONFIRM` stays closed.


In [ ]:
import os, sys, subprocess
from pathlib import Path

if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/Research-phase3-e44')
CODE_REF = 'phase3'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
# Visible Stage A2 E44 settings.
MODEL = 'qwen'  # 'qwen' or 'gemma'
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
PLAN_REF = None  # Paste {'path': ..., 'sha256': ...} or path to phase3/plans/.../_COMPLETE.json
RESUME_ANALYSIS_REF = None  # Optional explicit completed E44 analysis ref


In [ ]:
import json

if MODEL not in ('qwen', 'gemma'):
    raise RuntimeError(f"MODEL must be 'qwen' or 'gemma'; got {MODEL!r}")
if PLAN_REF is None:
    raise RuntimeError('Set PLAN_REF in Cell 3 before running E44.')
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Phase 3 checkout.')
if not PHASE1_ROOT.is_dir():
    raise RuntimeError(f'Missing mounted PHASE1_ROOT directory: {PHASE1_ROOT}')

plan_arg = json.dumps(PLAN_REF) if isinstance(PLAN_REF, dict) else str(PLAN_REF)
cmd = [
    sys.executable, '-u', '-m', 'src.phase3.run_e44',
    '--phase1-root', str(PHASE1_ROOT),
    '--plan', plan_arg,
    '--model', MODEL,
    '--storage-scope', 'persistent-drive',
]
if RESUME_ANALYSIS_REF is not None:
    resume_arg = json.dumps(RESUME_ANALYSIS_REF) if isinstance(RESUME_ANALYSIS_REF, dict) else str(RESUME_ANALYSIS_REF)
    cmd.extend(['--resume', resume_arg])

proc = subprocess.Popen(cmd, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
blockers = []
last_complete = None
for line in proc.stdout:
    print(line, end='', flush=True)
    try:
        payload = json.loads(line)
        if isinstance(payload, dict):
            if payload.get('status') == 'blocked':
                blockers.extend(payload.get('blockers', []))
            if payload.get('event') == 'complete':
                last_complete = payload
    except json.JSONDecodeError:
        pass
code = proc.wait()
if code != 0 or last_complete is None:
    detail = '; '.join(blockers) if blockers else f'exit code {code}'
    raise RuntimeError(f'Phase 3 E44 ({MODEL}) failed ({code}): {detail}')

print(f'\n=== AUTHORITATIVE PHASE 3 E44 ({MODEL}) ANALYSIS REFERENCE ===')
print(json.dumps(last_complete['analysis_ref'], indent=2))
print('=== PHASE 1 E19 PARITY ===')
print(json.dumps(last_complete['phase1_e19_parity'], indent=2))
print('=== DECISION SUMMARY (NOMINAL 0.75) ===')
print(json.dumps(last_complete['decision_summary'], indent=2))
